In [ ]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

silver_table = "nyc_taxi.silver.green_taxi"
silver_df = spark.table(silver_table)
duplicates = spark.table("nyc_taxi.quarantine.taxi_trips_duplicates_latest").withColumn(
    "trip_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("lpep_pickup_datetime"),
            F.col("lpep_dropoff_datetime"),
            F.col("PULocationID"),
            F.col("DOLocationID"),
        ),
        256,
    ),
)

duplicates_ids = duplicates.select("trip_id").distinct()
clean_df = silver_df.join(duplicates_ids, "trip_id", "left_anti")

# Replace the Silver snapshot after removing quarantined duplicate IDs.
clean_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)
